<a href="https://colab.research.google.com/github/cerebrunmedsdk-max/MRI-Metabolism-Mapper/blob/main/CMRO2_OEF_CTH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# [Step 0] DICOM to NIfTI Conversion
# =========================================================
import os
import dicom2nifti

def convert_dicom_to_nifti(patient_ids, base_dir, base_nifti_dir):
    print(">>> [0/3] Starting DICOM to NIfTI conversion...")

    modalities = {
        'T1': 't1_raw',
        'MPASL_pre': 'mpasl_base',
        'MPASL_post': 'diampasl_post',
        'CMRO2': 'cmro2_map',
        'OEF': 'oef_map',
        'CTH': 'cth_map'
    }

    for pid in patient_ids:
        print(f" [Process] Converting DICOMs for Patient [{pid}]...")
        patient_dicom_dir = os.path.join(base_dir, pid)
        patient_nifti_dir = os.path.join(base_nifti_dir, pid, 'NIfTI_Output')
        os.makedirs(patient_nifti_dir, exist_ok=True)

        for folder_name, file_prefix in modalities.items():
            dicom_path = os.path.join(patient_dicom_dir, folder_name)
            nifti_filename = f"{file_prefix}_{pid}.nii.gz"
            nifti_path = os.path.join(patient_nifti_dir, nifti_filename)

            if os.path.exists(dicom_path):
                if not os.path.exists(nifti_path):
                    try:
                        dicom2nifti.dicom_series_to_nifti(dicom_path, nifti_path, reorient_nifti=True)
                        print(f"    [Success] Converted {folder_name} -> {nifti_filename}")
                    except Exception as e:
                        print(f"    [Error] Failed to convert {folder_name}: {e}")
                else:
                    print(f"    [Skipped] Already exists: {nifti_filename}")
            else:
                print(f"    [Warning] DICOM folder not found: {dicom_path}")

# =========================================================
# [Step 1] Essential Libraries & Environment Setup
# =========================================================
print(">>> [1/3] Loading essential libraries...")
!pip install -q antspyx nibabel matplotlib pandas dicom2nifti scipy openpyxl

import os
import numpy as np
import pandas as pd
import ants as antspy
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.stats import skew, kurtosis
from google.colab import drive

# Mount Google Drive if running in Google Colab environment
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

def load_rpi(path):
    """Loads an image and reorients it to RPI (Right-Posterior-Inferior) space."""
    return antspy.reorient_image2(antspy.image_read(path), 'RPI')

# =========================================================
# [Step 2] Patient List & Analysis Configuration
# =========================================================
base_dir = '/content/drive/MyDrive'

patient_ids = ['2', '4', '15', '49', '53', '57', '59', '67', '103', '107', '111', '127']
ANALYSIS_MODE = 'CMRO2'

excel_path = os.path.join(base_dir, f'All_Patients_Ultimate_{ANALYSIS_MODE}_Analysis.xlsx')

print(f"\n>>> [2/3] Initializing [{ANALYSIS_MODE}] pipeline for {len(patient_ids)} patients.")
print(f"    (Active Parameters: HD-BET+ITK-SNAP Masking, Supratentorial ROI, 2px Erosion, 10% Core Exclusion)")

# Convert DICOM to NIfTI and save in Patient ID folder(base_dir/pid/)
convert_dicom_to_nifti(patient_ids, base_dir, base_dir)

# =========================================================
# [Step 3] Batch Processing Loop
# =========================================================
for pid_str in patient_ids:
    print(f"\n{'='*60}")
    print(f" [Processing] Patient ID: {pid_str} | Mode: {ANALYSIS_MODE}")

    patient_base_dir = os.path.join(base_dir, pid_str)
    nifti_out_dir = os.path.join(patient_base_dir, 'NIfTI_Output')
    os.makedirs(nifti_out_dir, exist_ok=True)

    # ---------------------------------------------------------
    # Skip completed patients
    # ---------------------------------------------------------
    final_qc_path = os.path.join(nifti_out_dir, f'Patient_{pid_str}_QC_Trimmed_{ANALYSIS_MODE}.png')
    if os.path.exists(final_qc_path):
        print(f"  [Skipped] Patient [{pid_str}] already processed. Output exists.")
        continue
    # ---------------------------------------------------------

    target_nifti_name = f"{ANALYSIS_MODE.lower()}_map_{pid_str}.nii.gz"
    target_nifti_path = os.path.join(nifti_out_dir, target_nifti_name)

    t1_nifti_path = os.path.join(nifti_out_dir, f't1_raw_{pid_str}.nii.gz')
    mpasl_nifti_path = os.path.join(nifti_out_dir, f'mpasl_base_{pid_str}.nii.gz')
    diampasl_nifti_path = os.path.join(nifti_out_dir, f'diampasl_post_{pid_str}.nii.gz')

    # Explicitly defining the HD-BET + ITK-SNAP refined mask
    # This mask represents T1 data after HD-BET extraction and manual removal of residual skull, and cavernous sinus mapping via ITK-SNAP
    # This ROI mask represents T1 data after addiional removal of brainstem.
    hdbet_refined_mask_path = os.path.join(nifti_out_dir, f'hdbet_itksnap_refined_mask_{pid_str}.nii.gz')
    roi_mask_path = os.path.join(nifti_out_dir, f'roi_supratentorial_{pid_str}.nii.gz')

    # Verify presence of all requisite files
    required_files = [t1_nifti_path, target_nifti_path, mpasl_nifti_path, diampasl_nifti_path, hdbet_refined_mask_path, roi_mask_path]
    if not all(os.path.exists(p) for p in required_files):
        print(f"  [Skipped] Missing required inputs. Please verify {hdbet_refined_mask_path} or {roi_mask_path}.")
        continue

    # Precise Registration & Brain Tissue Extraction
    fi = load_rpi(target_nifti_path)
    mi_t1 = load_rpi(t1_nifti_path)
    mi_mpasl = load_rpi(mpasl_nifti_path)
    mi_diampasl = load_rpi(diampasl_nifti_path)

    print(f"  -> Loading High-Fidelity Mask (HD-BET extraction + ITK-SNAP manual refinement)...")
    refined_mask = load_rpi(hdbet_refined_mask_path)

    voxel_vol_ml = (fi.spacing[0] * fi.spacing[1] * fi.spacing[2]) / 1000.0

    # N4 Bias Field Correction & Applying the refined mask for K-Means segmentation
    mi_t1_n4 = antspy.n4_bias_field_correction(mi_t1)
    mask_sync = antspy.resample_image_to_target(image=refined_mask, target=mi_t1_n4, interp_type='nearestNeighbor')
    mask_sync = antspy.threshold_image(mask_sync, low_thresh=0.5, high_thresh=100.0, inval=1, outval=0)

    # Segmenting the rigorously stripped T1 image to isolate pure parenchyma
    seg = antspy.kmeans_segmentation(mi_t1_n4, k=2, kmask=mask_sync)
    auto_tissue_mask = antspy.threshold_image(seg['segmentation'], low_thresh=2, high_thresh=2)

    print(f"  -> Executing {ANALYSIS_MODE} Non-linear Registration (SyN)...")
    reg_syn = antspy.registration(fixed=fi, moving=mi_t1_n4 * auto_tissue_mask, type_of_transform='SyN')
    reg_mpasl = antspy.registration(fixed=fi, moving=mi_mpasl, type_of_transform='Rigid')
    reg_diampasl = antspy.registration(fixed=fi, moving=mi_diampasl, type_of_transform='Rigid')

    # Morphological Operations: Supratentorial ROI and 2px Erosion
    roi_mask = load_rpi(roi_mask_path)
    roi_in_target = antspy.apply_transforms(fixed=fi, moving=roi_mask, transformlist=reg_syn['fwdtransforms'], interpolation='nearestNeighbor')
    eroded_roi = antspy.morphology(roi_in_target, operation='erode', radius=2, mtype='binary')
    eroded_roi_np = (eroded_roi.numpy() > 0)

    # CVR-based Target Masking
    mpasl_data = reg_mpasl['warpedmovout'].numpy()
    diampasl_data = reg_diampasl['warpedmovout'].numpy()
    target_map_data = fi.numpy()

    pure_parenchyma_mask = (antspy.apply_transforms(fixed=fi, moving=auto_tissue_mask, transformlist=reg_syn['fwdtransforms'], interpolation='nearestNeighbor').numpy() > 0)
    base_median = np.median(mpasl_data[pure_parenchyma_mask & (mpasl_data > 0)])
    cvr_map = ((diampasl_data - mpasl_data) / base_median) * 100

    preserved_mask = pure_parenchyma_mask & (cvr_map >= 10) & (target_map_data > 0) & eroded_roi_np
    exhausted_mask = pure_parenchyma_mask & (cvr_map <= 0) & (target_map_data > 0) & eroded_roi_np
    gray_zone_mask = pure_parenchyma_mask & (cvr_map > 0) & (cvr_map < 10) & (target_map_data > 0) & eroded_roi_np

    # Data Extraction & Modality-Specific Standardization
    ref_vals = target_map_data[preserved_mask].astype(np.float64)
    tar_vals = target_map_data[exhausted_mask].astype(np.float64)

    if len(ref_vals) < 50 or len(tar_vals) < 50:
        print(f"  [Skipped] Insufficient voxel count in target regions.")
        continue

    clean_ref = ref_vals[ref_vals <= np.percentile(ref_vals, 99)]
    ref_median = np.median(clean_ref)

    if ANALYSIS_MODE in ['CMRO2', 'CTH']:
        final_target = (tar_vals / ref_median) * 100 if ref_median > 0 else tar_vals
        val_unit = "(Relative %)"
    elif ANALYSIS_MODE == 'OEF':
        final_target = tar_vals
        val_unit = "(Absolute Raw)"

    # Physiological Trimming (Core/Noise Exclusion)
    pre_count = len(final_target)
    if ANALYSIS_MODE == 'CMRO2':
        final_target = final_target[(final_target > 10) & (final_target <= 250)]
    elif ANALYSIS_MODE == 'OEF':
        final_target = final_target[(final_target > 0.05) & (final_target <= 0.8)]
    elif ANALYSIS_MODE == 'CTH':
        final_target = final_target[final_target > 0]
    post_count = len(final_target)

    if pre_count > post_count:
        print(f"  -> Outlier exclusion applied: {pre_count - post_count} voxels removed.")

    if len(final_target) < 50:
        print(f"  [Skipped] Insufficient valid voxels post-trimming.")
        continue

    # Statistical Extraction (10th to 90th Percentiles)
    p10, p25, median_val, p75, p90 = np.percentile(final_target, [10, 25, 50, 75, 90])

    # Save outputs to Excel
    res = {
        'Patient_ID': pid_str,
        'Ref_Vol_mL': len(clean_ref) * voxel_vol_ml,
        'Target_Vol_mL': len(final_target) * voxel_vol_ml,
        'Ref_Median_Baseline': ref_median,
        'Target_Median(50th)': median_val,
        'Skewness': skew(final_target), 'Kurtosis': kurtosis(final_target),
        '10th_Pct': p10, '25th_Pct': p25, '75th_Pct': p75, '90th_Peak': p90
    }

    if not os.path.exists(excel_path):
        pd.DataFrame([res]).to_excel(excel_path, index=False)
    else:
        pd.concat([pd.read_excel(excel_path), pd.DataFrame([res])], ignore_index=True)\
          .drop_duplicates(subset=['Patient_ID'], keep='last')\
          .to_excel(excel_path, index=False)

    # Quality Control (QC) Visualization Generation
    print(f"  -> Generating spatial and histogram QC figures...")
    vis_data = np.zeros_like(target_map_data)
    vis_data[preserved_mask], vis_data[gray_zone_mask], vis_data[exhausted_mask] = 1, 2, 3
    cmap_cvr = ListedColormap(['none', 'dodgerblue', 'gold', 'crimson'])

    fig = plt.figure(figsize=(20, 18))
    gs = fig.add_gridspec(nrows=7, ncols=8)
    for i in range(min(fi.shape[2], 48)):
        r, c = i // 8, i % 8
        ax = fig.add_subplot(gs[r, c])
        ax.imshow(np.rot90(target_map_data[:, :, i], k=1), cmap='gray')
        if np.any(vis_data[:, :, i] > 0):
            ax.imshow(np.rot90(vis_data[:, :, i], k=1), cmap=cmap_cvr, alpha=0.55, vmin=0, vmax=3)
        if np.any(eroded_roi_np[:, :, i]):
             ax.contour(np.rot90(eroded_roi_np[:, :, i], k=1), colors='cyan', linewidths=0.5, levels=[0.5], linestyles='dotted')
        ax.axis('off')

    ax_hist = fig.add_subplot(gs[6, 1:-1])

    if ANALYSIS_MODE == 'CMRO2':
        bins_setting = np.arange(10, 251, 2.5)
    elif ANALYSIS_MODE == 'OEF':
        bins_setting = np.arange(0.05, 0.85, 0.01)
    else:
        bins_setting = 100

    ax_hist.hist(final_target, bins=bins_setting, color='crimson', alpha=0.7, edgecolor='black')

    ref_line = 100 if ANALYSIS_MODE in ['CMRO2', 'CTH'] else ref_median
    ax_hist.axvline(ref_line, color='dodgerblue', linewidth=3, label='Reference Baseline')
    ax_hist.axvline(median_val, color='orange', linestyle='dashed', linewidth=2, label=f'Median: {median_val:.2f}')

    bad_val = p10 if ANALYSIS_MODE == 'CMRO2' else p90
    bad_label = '10th Pct Drop' if ANALYSIS_MODE == 'CMRO2' else '90th Pct Peak'
    ax_hist.axvline(bad_val, color='purple', linestyle='dashed', linewidth=2, label=f'{bad_label}: {bad_val:.2f}')

    ax_hist.set_title(f"Target {ANALYSIS_MODE} Distribution (Physiologically Trimmed)")
    ax_hist.set_xlabel(f"{ANALYSIS_MODE} Value {val_unit}")
    ax_hist.set_ylabel("Voxel Count")
    ax_hist.legend()

    plt.savefig(os.path.join(nifti_out_dir, f'Patient_{pid_str}_QC_Trimmed_{ANALYSIS_MODE}.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f">>> [Complete] Patient [{pid_str}] processing and core trimming finished successfully.")